# **Multi-Metric Sales Dashboard**

In [ ]:
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import matplotlib.pyplot as plt

In [ ]:
# Sample sales data for 3 products over 12 months
data = pd.DataFrame({
    "Date": pd.date_range(start="2025-01-01", periods=12, freq="M").tolist()*3,
    "Product": ["Product A"]*12 + ["Product B"]*12 + ["Product C"]*12,
    "Units Sold": [20, 25, 30, 22, 28, 35, 40, 38, 45, 50, 48, 55]*3,
    "Price": [10]*36,
    "Discount": [0, 5, 10, 0, 5, 10, 0, 5, 10, 0, 5, 10]*3
})

# Compute metrics
data["Revenue"] = data["Units Sold"] * data["Price"]
data["Profit"] = data["Revenue"] - (data["Revenue"] * data["Discount"] / 100)
data["Month"] = data["Date"].dt.strftime("%B")

/tmp/ipython-input-1912944406.py:3: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  "Date": pd.date_range(start="2025-01-01", periods=12, freq="M").tolist()*3,


In [ ]:
product_dropdown = widgets.Dropdown(
    options=["All Products"] + list(data["Product"].unique()),
    description="Select Product:",
    style={'description_width': 'initial'}
)

metric_dropdown = widgets.Dropdown(
    options=["Revenue", "Units Sold", "Discount", "Profit"],
    description="Select Metric:",
    style={'description_width': 'initial'}
)

month_dropdown = widgets.Dropdown(
    options=["All Months"] + list(data["Month"].unique()),
    description="Select Month:",
    style={'description_width': 'initial'}
)

output = widgets.Output()

In [ ]:
def show_sales_dashboard(b):
    with output:
        clear_output()

        product = product_dropdown.value
        metric = metric_dropdown.value
        month = month_dropdown.value

        # Filter data
        df = data.copy()
        if product != "All Products":
            df = df[df["Product"] == product]
        if month != "All Months":
            df = df[df["Month"] == month]

        # Total metric
        total = df[metric].sum()
        display(HTML(f"<h3 style='color:#4CAF50'>Total {metric}: {total}</h3>"))

        # Color-coded advice
        advice = ""
        if metric in ["Units Sold", "Revenue", "Profit"]:
            if total < 50:
                advice = "<p style='color:red'>Advice: Consider promotions or strategy improvement.</p>"
            elif total < 150:
                advice = "<p style='color:orange'>Advice: Moderate performance, can improve.</p>"
            else:
                advice = "<p style='color:green'>Advice: Excellent performance!</p>"
        else:  # Discount
            if total > 20:
                advice = "<p style='color:red'>Advice: Discounts too high, reduce for profit.</p>"
            else:
                advice = "<p style='color:green'>Discounts are optimal.</p>"
        display(HTML(advice))

        # Plot trend
        monthly_data = df.groupby("Date")[metric].sum().reset_index()
        plt.figure(figsize=(8,4))
        plt.plot(monthly_data["Date"], monthly_data[metric], marker='o', color='#2196F3')
        plt.title(f"{metric} Trend")
        plt.xlabel("Date")
        plt.ylabel(metric)
        plt.xticks(rotation=45)
        plt.grid(True)
        plt.show()

        # Top products if viewing all products
        if product == "All Products":
            top_products = df.groupby("Product")[metric].sum().sort_values(ascending=False)
            display(HTML("<b>Top Products:</b>"))
            for p, val in top_products.items():
                display(HTML(f"<p>{p}: {val}</p>"))

In [ ]:
update_button = widgets.Button(description="Show Dashboard", button_style="success")
update_button.on_click(show_sales_dashboard)

In [ ]:
dashboard_ui = widgets.VBox([
    widgets.HTML("<h1 style='color:#4CAF50'>Multi-Metric Sales Dashboard</h1>"),
    product_dropdown,
    month_dropdown,
    metric_dropdown,
    update_button,
    output
])

display(dashboard_ui)